In [1]:
# !wget https://data.ncl.ac.uk/ndownloader/articles/24574753/versions/1 -O ../data/Gutenberg.zip
# !mkdir ../data/Gutenberg
# !unzip ../data/Gutenberg.zip -d ../data/Gutenberg/
# !rm ../data/Gutenberg.zip

In [2]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split, TensorDataset, Subset
from torchvision import datasets, transforms
from functools import partial
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
    torch.set_deterministic_debug_mode(True)



################################
#     RESTART     RUNTIME      #
################################
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
from ssonn.train.train import *

In [3]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=27 * 18, hidden_size = 16, output_size=6):
        super(SimpleFCN, self).__init__()
        # self.fc_input = nn.Linear(input_size, hidden_size)
        # self.act = nn.ReLU()
        self.fc_output = nn.Linear(input_size, output_size)


    def forward(self, x):
        # x = self.fc_input(x)
        # x = self.act(x)
        x = self.fc_output(x)
        return x

In [4]:
hyperparams = {
    "num_epochs": 256,
    "batch_size": 256,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {"fc_output": 0.1},
    "pruning_thresholds": {"fc_output": 1},
    "plateau_threshold": 0.002,
    "min_epochs_between_expansions": 16,
    "plateau_window_size": 5,
    "learning_rate": 1e-1,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 2000,
    "weight_decay": 0,
    "diff_threshold": 0.05,
    "t_max": 32,
    "warmup_epochs": 5
}


In [5]:
train_x = np.load("../data/Gutenberg/train_x.npy")
train_y = np.load("../data/Gutenberg/train_y.npy")
valid_x = np.load("../data/Gutenberg/valid_x.npy")
valid_y = np.load("../data/Gutenberg/valid_y.npy")
test_x = np.load('../data/Gutenberg/test_x.npy')
test_y = np.load('../data/Gutenberg/test_y.npy')

train_tensor_x = torch.tensor(train_x).float().view(-1, 27 * 18)
train_tensor_y = torch.tensor(train_y).long()

valid_tensor_x = torch.tensor(valid_x).float().view(-1, 27 * 18)
valid_tensor_y = torch.tensor(valid_y).long()

test_tensor_x = torch.tensor(test_x).float().view(-1, 27 * 18)
test_tensor_y = torch.tensor(test_y).long()

train_dataset = TensorDataset(train_tensor_x, train_tensor_y)
valid_dataset = TensorDataset(valid_tensor_x, valid_tensor_y)
test_dataset = TensorDataset(test_tensor_x, test_tensor_y)

# Новая логика: finder — доля train для расширения (100% = полный train)
finder_fraction = 1.0
n_finder = max(1, int(finder_fraction * len(train_dataset)))
finder_dataset = Subset(train_dataset, range(n_finder))

bs = hyperparams['batch_size']
train_loader = DataLoader(train_dataset, batch_size=bs, shuffle=True, generator=g)
finder_loader = DataLoader(finder_dataset, batch_size=bs, shuffle=False)
val_loader = DataLoader(valid_dataset, batch_size=bs, shuffle=False, generator=g)
test_loader = DataLoader(test_dataset, batch_size=bs, shuffle=False, generator=g)

In [6]:
train_tensor_y[:10]

tensor([2, 2, 3, 5, 0, 2, 0, 3, 3, 1])

In [7]:
base_model = SimpleFCN()
model = convert_to_scaled_net(base_model, layers_to_convert=[base_model.fc_output], device=device)

model.fc_output.dropout.p = 0.3

Converting layer 'fc_output' to ScaledNet...


In [8]:
from comet_ml import start
from comet_ml.integration.pytorch import log_model
experiment = start(
    api_key=os.environ.get("COMET_API_KEY"),
    workspace=os.environ.get("COMET_WORKSPACE"),
    project_name="gutenberg",
)
experiment.log_parameters(hyperparams)
experiment.set_name("gutenberg origin")

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch, sklearn.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/vanya9090/gutenberg/14c7a77719374958b8a241ada8d3e48e



In [9]:
criterion = nn.CrossEntropyLoss()

def optimizer_factory(model_instance, current_lr=None, base_lr=None):
    if base_lr is None:
        base_lr = hyperparams['learning_rate']
    if current_lr is None:
        current_lr = base_lr
    
    new_optimizer = optim.SGD(
        model_instance.parameters(),
        lr=current_lr,
        weight_decay=hyperparams['weight_decay'],
        momentum=0.9
    )
    
    for group in new_optimizer.param_groups:
        group['initial_lr'] = base_lr 
        
    return new_optimizer

trained_model = train_ssonn(
    model,
    train_loader,
    val_loader,
    finder_loader,
    criterion,
    optimizer_factory,
    hyperparams,
    device,
    experiment,
    True,
)
sparse_model = trained_model

--- Starting SSONN Training ---


Epoch 1/256 | T-Loss: 1.7365 | V-Loss: 1.7172 | Acc: 0.2782 | LRs: [1.0e-01] | Params: 2922/2922


Epoch 2/256 | T-Loss: 1.6919 | V-Loss: 1.7003 | Acc: 0.2875 | LRs: [9.9e-02] | Params: 2922/2922


Epoch 3/256 | T-Loss: 1.6789 | V-Loss: 1.6939 | Acc: 0.2897 | LRs: [9.8e-02] | Params: 2922/2922


Epoch 4/256 | T-Loss: 1.6707 | V-Loss: 1.6855 | Acc: 0.2984 | LRs: [9.6e-02] | Params: 2922/2922


Epoch 5/256 | T-Loss: 1.6654 | V-Loss: 1.6854 | Acc: 0.2977 | LRs: [9.4e-02] | Params: 2922/2922


... (74 outputs left out)

Epoch 79/256 | T-Loss: 0.5765 | V-Loss: 1.8467 | Acc: 0.4014 | LRs: [5.0e-02] | Params: 425204/751789


Epoch 80/256 | T-Loss: 0.5512 | V-Loss: 1.8666 | Acc: 0.4032 | LRs: [4.5e-02] | Params: 425204/751789


Epoch 81/256 | T-Loss: 0.5259 | V-Loss: 1.8741 | Acc: 0.4059 | LRs: [4.0e-02] | Params: 425204/751789


KeyboardInterrupt: 

In [ ]:
experiment.end()

In [ ]:
_, accuracy = eval_one_epoch(sparse_model, criterion, test_loader, hyperparams['task_type'], device)
params = get_params_amount(sparse_model)

In [ ]:
accuracy, params

In [43]:
torch.save(sparse_model, 'gutenberg.pt')